# 📘 Khóa học: HR Analyst với SQL + PostgreSQL (7 buổi)

## 🎯 Tổng quan chương trình

- **Buổi 1:** 🚀 Giới thiệu dự án
- **Buổi 2:** 🗄️ Schema & Setup database
- **Buổi 3:** 👥 Employee Overview Analysis
- **Buổi 4:** 📊 Turnover Analysis
- **Buổi 5:** 💰 Salary Analysis
- **Buổi 6:** 📈 Engagement & Advanced Analysis
- **Buổi 7:** ✅ Quiz HR Analytics SQL

---

# 💰 Buổi 5: Salary Analysis

## 🎯 Mục tiêu buổi này
✅ Sử dụng NUMERIC và ROUND để làm tròn số liệu lương

✅ Phân tích phân bố lương với GROUP BY và aggregate functions

✅ Tính toán khoảng cách lương theo giới tính (gender pay gap)

✅ So sánh lương theo cấp bậc và phòng ban

✅ Giới thiệu Window Functions và CTEs (Common Table Expressions)

✅ Thực hành 5 bài tập cơ bản + 4 bài nâng cao với đáp án chi tiết

✅ Ôn tập 5 quiz từ cơ bản → nâng cao

---

## 🌏 Đề tài Salary Analysis là gì?

**Salary Analysis** là phân tích mức lương và cấu trúc lương trong tổ chức. Đây là một phần quan trọng của HR Analytics, giúp doanh nghiệp:

- 💵 **Đánh giá công bằng**: Lương có công bằng không?
- 📈 **Tối ưu chi phí**: Chi phí nhân sự hiệu quả
- 🎯 **Thu hút nhân tài**: Lương cạnh tranh trên thị trường
- 📊 **Dự đoán turnover**: Lương thấp dẫn đến nghỉ việc

### 📊 Ví dụ điển hình ở Việt Nam (2026)
- **Ngành Tech**: Lương trung bình 25-40 triệu/tháng, gender gap 5-10%
- **Ngành Tài chính**: 30-50 triệu/tháng, nhưng gender gap cao hơn
- **Ngành Bán lẻ**: 8-15 triệu/tháng, tập trung vào thưởng hiệu suất

**Tác động kinh tế**: Gender pay gap 1% = giảm 0.5% năng suất. Chi phí lương chiếm 60-70% chi phí nhân sự.

### ❓ Các truy vấn thường được hỏi
- Lương trung bình theo phòng ban
- Gender pay gap trong công ty
- Phân bố lương theo cấp bậc
- Tăng lương hàng năm

---

## 📚 Lý thuyết: Window Functions

**Window Functions** là các hàm tính toán trên một "cửa sổ" (window) của dữ liệu, không làm thay đổi số lượng rows như GROUP BY.

### Cú pháp cơ bản:
```sql
SELECT column1, column2,
       window_function() OVER (PARTITION BY partition_column ORDER BY order_column) AS alias
FROM table_name;
```

### Các hàm phổ biến:
- **ROW_NUMBER()**: Gán số thứ tự
- **RANK()**: Xếp hạng (có thể trùng)
- **DENSE_RANK()**: Xếp hạng liên tục
- **LAG()**: Lấy giá trị của row trước
- **LEAD()**: Lấy giá trị của row sau
- **SUM() OVER()**: Tổng tích lũy
- **AVG() OVER()**: Trung bình trong cửa sổ

### Ví dụ:
```sql
SELECT emp_id, base_salary,
       ROW_NUMBER() OVER (ORDER BY base_salary DESC) AS salary_rank
FROM salaries;
```

---

## 📚 Lý thuyết: CTEs (Common Table Expressions)

**CTEs** là bảng tạm thời được định nghĩa trong câu query, giúp code dễ đọc và tái sử dụng.

### Cú pháp:
```sql
WITH cte_name AS (
    SELECT column1, column2
    FROM table_name
    WHERE condition
)
SELECT * FROM cte_name;
```

### Ưu điểm:
- Dễ debug: Chia query thành các phần nhỏ
- Tái sử dụng: Có thể JOIN với chính nó
- Đọc hiểu: Như subquery nhưng rõ ràng hơn

### Ví dụ:
```sql
WITH avg_salary AS (
    SELECT dept_id, ROUND(AVG(base_salary),2) AS dept_avg
    FROM employees e
    JOIN salaries s ON e.emp_id = s.emp_id
    GROUP BY dept_id
)
SELECT * FROM avg_salary;
```

---

## 🔍 BƯỚC 1: SELECT Cơ bản - Xem dữ liệu lương

**Lương** thường được lưu trong bảng `salaries`. Hãy bắt đầu bằng cách xem cấu trúc dữ liệu.

### 1.1 Xem lương hiện tại của nhân viên
```sql
SELECT e.emp_id, e.first_name, e.last_name, s.base_salary + COALESCE(s.bonus, 0) AS total_salary, s.effective_date
FROM employees e
JOIN salaries s ON e.emp_id = s.emp_id
WHERE s.effective_date = (SELECT MAX(effective_date) FROM salaries s2 WHERE s2.emp_id = e.emp_id)
ORDER BY total_salary DESC
LIMIT 10;
```
**Kỳ vọng:** Hiển thị 10 nhân viên có lương cao nhất với thông tin cơ bản.

### 1.2 Lương trung bình của công ty
```sql
SELECT ROUND(AVG(s.base_salary + COALESCE(s.bonus, 0)), 0) as avg_salary
FROM salaries s
WHERE s.effective_date = (SELECT MAX(effective_date) FROM salaries s2 WHERE s2.emp_id = s.emp_id);
```
**Kỳ vọng:** Một số làm tròn, ví dụ: 25000000 (25 triệu VND).

---

## 🔍 BƯỚC 2: ROUND và NUMERIC - Làm tròn số liệu

**ROUND** dùng để làm tròn số, **NUMERIC** để định dạng số chính xác.

### 2.1 Làm tròn lương đến hàng nghìn
```sql
SELECT emp_id, ROUND((base_salary + COALESCE(bonus, 0)) / 1000, 0) * 1000 as salary_rounded
FROM salaries
WHERE effective_date = (SELECT MAX(effective_date) FROM salaries s2 WHERE s2.emp_id = salaries.emp_id);
```
**Kỳ vọng:** Lương được làm tròn đến 1000 VND gần nhất.

### 2.2 Tính lương hàng tháng (giả sử lương năm)
```sql
SELECT emp_id, ROUND((base_salary + COALESCE(bonus, 0)) / 12, 2) as monthly_salary
FROM salaries
WHERE effective_date = (SELECT MAX(effective_date) FROM salaries s2 WHERE s2.emp_id = salaries.emp_id);
```
**Kỳ vọng:** Lương chia 12 và làm tròn 2 chữ số thập phân.

---

## 🔍 BƯỚC 3: Phân tích phân bố lương

Sử dụng GROUP BY để nhóm theo phòng ban, cấp bậc.

### 3.1 Lương trung bình theo phòng ban
```sql
SELECT d.dept_name, ROUND(AVG(s.base_salary + COALESCE(s.bonus, 0)), 0) as avg_salary, COUNT(*) as employee_count
FROM employees e
JOIN salaries s ON e.emp_id = s.emp_id
JOIN departments d ON e.dept_id = d.dept_id
WHERE s.effective_date = (SELECT MAX(effective_date) FROM salaries s2 WHERE s2.emp_id = e.emp_id)
GROUP BY d.dept_name
ORDER BY avg_salary DESC;
```
**Kỳ vọng:** Bảng với dept_name, avg_salary, employee_count.

### 3.2 Phân bố lương theo cấp bậc (từ job_titles)
```sql
SELECT jt.title_name, ROUND(AVG(s.base_salary + COALESCE(s.bonus, 0)), 0) as avg_salary, MIN(s.base_salary + COALESCE(s.bonus, 0)) as min_salary, MAX(s.base_salary + COALESCE(s.bonus, 0)) as max_salary
FROM employees e
JOIN salaries s ON e.emp_id = s.emp_id
JOIN job_titles jt ON e.title_id = jt.title_id
WHERE s.effective_date = (SELECT MAX(effective_date) FROM salaries s2 WHERE s2.emp_id = e.emp_id)
GROUP BY jt.title_name
ORDER BY jt.title_name;
```
**Kỳ vọng:** Bảng với title_name, avg/min/max salary.

---

## 🔍 BƯỚC 4: Gender Pay Gap

Tính khoảng cách lương giữa nam và nữ.

### 4.1 Lương trung bình theo giới tính
```sql
SELECT e.gender, ROUND(AVG(s.base_salary + COALESCE(s.bonus, 0)), 0) as avg_salary, COUNT(*) as count
FROM employees e
JOIN salaries s ON e.emp_id = s.emp_id
WHERE s.effective_date = (SELECT MAX(effective_date) FROM salaries s2 WHERE s2.emp_id = e.emp_id)
GROUP BY e.gender;
```
**Kỳ vọng:** Bảng với gender (M/F), avg_salary, count.

### 4.2 Tính gender pay gap (%)
```sql
WITH gender_salary AS (
    SELECT e.gender, AVG(s.base_salary + COALESCE(s.bonus, 0)) as avg_salary
    FROM employees e
    JOIN salaries s ON e.emp_id = s.emp_id
    WHERE s.effective_date = (SELECT MAX(effective_date) FROM salaries s2 WHERE s2.emp_id = e.emp_id)
    GROUP BY e.gender
)
SELECT 
    ROUND(((SELECT avg_salary FROM gender_salary WHERE gender = 'M') - 
           (SELECT avg_salary FROM gender_salary WHERE gender = 'F')) / 
          (SELECT avg_salary FROM gender_salary WHERE gender = 'M') * 100, 2) as pay_gap_percent
FROM gender_salary
LIMIT 1;
```
**Kỳ vọng:** Một số phần trăm, ví dụ: 8.50 (8.5% gap).

---

## 📝 BÀI TẬP CƠ BẢN (5 bài)

### Bài 1: Lương cao nhất và thấp nhất
Viết query tìm lương cao nhất và thấp nhất trong công ty.

**Đáp án:**
```sql
SELECT MAX(base_salary + COALESCE(bonus, 0)) as max_salary,
       MIN(base_salary + COALESCE(bonus, 0)) as min_salary
FROM salaries
WHERE effective_date = (SELECT MAX(effective_date) FROM salaries s2 WHERE s2.emp_id = salaries.emp_id);
```
**Giải thích:** Sử dụng MAX và MIN để tìm giá trị cao nhất/thấp nhất. COALESCE để xử lý bonus NULL. Subquery để lấy lương hiện tại.

### Bài 2: Số lượng nhân viên theo khoảng lương
Đếm số nhân viên có lương < 20tr, 20-30tr, >30tr.

**Đáp án:**
```sql
SELECT 
    CASE 
        WHEN total_salary < 20000000 THEN '< 20M'
        WHEN total_salary BETWEEN 20000000 AND 30000000 THEN '20-30M'
        ELSE '> 30M'
    END as salary_range,
    COUNT(*) as count
FROM (
    SELECT emp_id, base_salary + COALESCE(bonus, 0) as total_salary
    FROM salaries
    WHERE effective_date = (SELECT MAX(effective_date) FROM salaries s2 WHERE s2.emp_id = salaries.emp_id)
) s
GROUP BY salary_range
ORDER BY salary_range;
```
**Giải thích:** Sử dụng CASE WHEN để phân loại khoảng lương. Subquery để tính total_salary. GROUP BY để đếm.

### Bài 3: Lương trung bình theo năm kinh nghiệm
Tính lương trung bình theo nhóm năm kinh nghiệm (0-2, 3-5, 6+).

**Đáp án:**
```sql
SELECT 
    CASE 
        WHEN EXTRACT(YEAR FROM AGE(CURRENT_DATE, hired_date)) <= 2 THEN '0-2 years'
        WHEN EXTRACT(YEAR FROM AGE(CURRENT_DATE, hired_date)) <= 5 THEN '3-5 years'
        ELSE '6+ years'
    END as experience_group,
    ROUND(AVG(s.base_salary + COALESCE(s.bonus, 0)), 0) as avg_salary
FROM employees e
JOIN salaries s ON e.emp_id = s.emp_id
WHERE s.effective_date = (SELECT MAX(effective_date) FROM salaries s2 WHERE s2.emp_id = e.emp_id)
GROUP BY experience_group
ORDER BY experience_group;
```
**Giải thích:** EXTRACT(YEAR FROM AGE()) để tính năm kinh nghiệm. CASE WHEN để nhóm. AVG và ROUND cho trung bình.

### Bài 4: Top 5 phòng ban có lương trung bình cao nhất
Liệt kê 5 phòng ban có lương trung bình cao nhất.

**Đáp án:**
```sql
SELECT d.dept_name, ROUND(AVG(s.base_salary + COALESCE(s.bonus, 0)), 0) as avg_salary
FROM employees e
JOIN salaries s ON e.emp_id = s.emp_id
JOIN departments d ON e.dept_id = d.dept_id
WHERE s.effective_date = (SELECT MAX(effective_date) FROM salaries s2 WHERE s2.emp_id = e.emp_id)
GROUP BY d.dept_name
ORDER BY avg_salary DESC
LIMIT 5;
```
**Giải thích:** JOIN 3 bảng. GROUP BY theo dept_name. ORDER BY avg_salary DESC, LIMIT 5.


## 📊 THÊM DỮ LIỆU MẪU CHO LỊCH SỬ LƯƠNG

Để tính tỷ lệ tăng lương hàng năm, bảng `salaries` cần có dữ liệu lịch sử. Dữ liệu hiện tại chỉ có 1 ngày áp dụng cho mỗi nhân viên. Hãy thêm dữ liệu mẫu cho 5 nhân viên với lương qua các năm.

### Thêm dữ liệu lịch sử lương
```sql
-- Thêm dữ liệu cho emp_id 1 (Nguyen Van A)
INSERT INTO salaries (emp_id, base_salary, bonus, effective_date) VALUES
(1, 22000000, 2500000, '2024-01-01'),
(1, 24000000, 3000000, '2025-01-01');

-- Thêm dữ liệu cho emp_id 2 (Tran Thi B)
INSERT INTO salaries (emp_id, base_salary, bonus, effective_date) VALUES
(2, 19500000, 1800000, '2024-01-01'),
(2, 21000000, 2000000, '2025-01-01');

-- Thêm dữ liệu cho emp_id 3 (Le Van C)
INSERT INTO salaries (emp_id, base_salary, bonus, effective_date) VALUES
(3, 27000000, 3500000, '2024-01-01'),
(3, 29000000, 4000000, '2025-01-01');

-- Thêm dữ liệu cho emp_id 4 (Pham Thi D)
INSERT INTO salaries (emp_id, base_salary, bonus, effective_date) VALUES
(4, 16500000, 1200000, '2024-01-01'),
(4, 18000000, 1500000, '2025-01-01');

-- Thêm dữ liệu cho emp_id 5 (Hoang Van E)
INSERT INTO salaries (emp_id, base_salary, bonus, effective_date) VALUES
(5, 33000000, 6000000, '2024-01-01'),
(5, 36000000, 7000000, '2025-01-01');
```

**Lưu ý:** Chạy các INSERT này trong PostgreSQL để thêm dữ liệu lịch sử. Sau đó, bài tập 5 sẽ hoạt động được.


### Bài 5: Lịch sử tăng/giảm lương của nhân viên
Cho mỗi nhân viên, liệt kê lương qua các năm và tính % thay đổi so với năm trước.

**Đáp án:**
```sql
WITH salary_stats AS (
  SELECT e.first_name || ' ' || e.last_name as employee_name,
         EXTRACT(YEAR FROM s.effective_date) as year,
         s.base_salary + COALESCE(s.bonus, 0) as total_salary,
         -- Tính % thay đổi
         ROUND(
           ( (s.base_salary + COALESCE(s.bonus, 0)) - 
             LAG(s.base_salary + COALESCE(s.bonus, 0)) OVER (PARTITION BY s.emp_id ORDER BY s.effective_date)
           )::numeric / 
           NULLIF(LAG(s.base_salary + COALESCE(s.bonus, 0)) OVER (PARTITION BY s.emp_id ORDER BY s.effective_date), 0) 
           * 100, 2
         ) as change_percent
  FROM employees e
  JOIN salaries s ON e.emp_id = s.emp_id
)
SELECT * 
FROM salary_stats 
WHERE change_percent IS NOT NULL;
```
**Giải thích:** 

1. LAG(s.base_salary + COALESCE(s.bonus, 0)) OVER (PARTITION BY s.emp_id ORDER BY s.effective_date)

- LAG(total_salary): Lấy giá trị của dòng ngay phía trên (tức là lương của lần trước đó).

- PARTITION BY e.emp_id: Chia dữ liệu theo từng nhân viên. Nó giúp đảm bảo bạn lấy lương cũ của đúng người đó, chứ không lấy nhầm lương của người khác ở dòng trên.

- ORDER BY s.effective_date: Sắp xếp theo thời gian để lệnh LAG lấy đúng lương của kỳ gần nhất, không bị lộn xộn.

2. NULLIF(..., 0)

- Đây là lệnh "phòng thủ" để tránh lỗi toán học.

- Vấn đề: Trong toán học, bạn không thể chia một số cho 0. Nếu một nhân viên mới vào có lương cũ bằng 0, phần mềm sẽ "văng lỗi" ngay lập tức.

- Cách hoạt động: NULLIF(giá_trị, 0) nói rằng: "Nếu giá trị này bằng 0, hãy biến nó thành NULL". Trong SQL, một số chia cho NULL sẽ ra NULL (êm đẹp) thay vì làm treo cả hệ thống.


---

## 🚀 BÀI TẬP NÂNG CAO (3 bài)

### Bài 6: Phân tích lương theo percentile
Tính lương ở percentile 25, 50, 75, 90.

**Đáp án:**
```sql
SELECT 
    PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY base_salary + COALESCE(bonus, 0)) as p25,
    PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY base_salary + COALESCE(bonus, 0)) as p50,
    PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY base_salary + COALESCE(bonus, 0)) as p75,
    PERCENTILE_CONT(0.90) WITHIN GROUP (ORDER BY base_salary + COALESCE(bonus, 0)) as p90
FROM salaries
WHERE effective_date = (SELECT MAX(effective_date) FROM salaries s2 WHERE s2.emp_id = salaries.emp_id);
```
**Giải thích:** PERCENTILE_CONT là hàm dùng để tìm giá trị tại một điểm phân vị cụ thể (ví dụ: trung vị, top 25%, top 75%) dựa trên một tập hợp số liệu liên tục.


Nó giúp bạn trả lời câu hỏi: "Mức lương ở ngưỡng giữa (50%) của công ty là bao nhiêu?" hoặc "Ngưỡng lương của top 10% nhân viên cao nhất là bao nhiêu?"

- Cấu trúc: PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY salary)

Với: 0.5: Điểm phân vị bạn muốn tìm (0.5 là 50% - tức là số trung vị). WITHIN GROUP (ORDER BY ...): Chỉ định tập dữ liệu nào và sắp xếp theo thứ tự nào để tìm điểm đó.


### Bài 7: Gender pay gap theo phòng ban
Tính gender pay gap cho từng phòng ban.

**Đáp án:**
```sql
WITH improved_dept_salary AS (
    -- Tính lương trung bình Nam và Nữ cùng lúc trên 1 dòng
    SELECT 
        d.dept_name,
        AVG(CASE WHEN e.gender = 'M' THEN s.base_salary + COALESCE(s.bonus, 0) END) as avg_male_salary,
        AVG(CASE WHEN e.gender = 'F' THEN s.base_salary + COALESCE(s.bonus, 0) END) as avg_female_salary
    FROM employees e
    JOIN salaries s ON e.emp_id = s.emp_id
    JOIN departments d ON e.dept_id = d.dept_id
    WHERE s.effective_date = (SELECT MAX(effective_date) FROM salaries s2 WHERE s2.emp_id = e.emp_id)
    GROUP BY d.dept_name
)
SELECT 
    dept_name,
    ROUND(((avg_male_salary - avg_female_salary) / NULLIF(avg_male_salary, 0) * 100)::numeric, 2) as pay_gap_percent
FROM improved_dept_salary
ORDER BY pay_gap_percent DESC;
```
**Giải thích:** CTE tính lương TB theo dept và gender. Subquery trong SELECT tính gap cho mỗi dept.

### Bài 8: Phân tích lương Remote/Hybrid/Onsite trong kỷ nguyên Work-from-Anywhere
Tính lương trung bình và chênh lệch giữa nhóm Remote, Hybrid và Onsite để phản ánh xu hướng hiện đại của môi trường làm việc.

**Đáp án:**
```sql
WITH latest_salary AS (
    SELECT e.emp_id, e.remote_status,
           s.base_salary + COALESCE(s.bonus, 0) AS total_salary,
           ROW_NUMBER() OVER (PARTITION BY e.emp_id ORDER BY s.effective_date DESC) AS rn
    FROM employees e
    JOIN salaries s ON e.emp_id = s.emp_id
)
SELECT 
    remote_status,
    COUNT(*) AS employee_count,
    ROUND(AVG(total_salary), 0) AS avg_salary,
    ROUND(MIN(total_salary), 0) AS min_salary,
    ROUND(MAX(total_salary), 0) AS max_salary
FROM latest_salary
WHERE rn = 1
GROUP BY remote_status
ORDER BY avg_salary DESC;
```
**Giải thích:**
- `ROW_NUMBER()` trong CTE chọn bản ghi lương mới nhất cho mỗi nhân viên.
- Phân nhóm theo `remote_status` (Remote/Hybrid/Onsite).
- Tính `AVG`, `MIN`, `MAX` lương cho mỗi mô hình làm việc.
- Cho phép so sánh nhanh: nhóm nào đang nhận lương cao hơn, nhóm nào cần chính sách cân bằng luơng hợp lý.


---

## ❓ QUIZ ÔN TẬP (5 câu)

### Quiz 1: Hàm nào dùng để làm tròn số?
A) ROUND 
B) NUMERIC 
C) AVG 
D) SUM

✅ Đáp án đúng: A) ROUND
Giải thích: `ROUND()` là hàm SQL dùng để làm tròn số, trong khi `NUMERIC` là kiểu dữ liệu.

### Quiz 2: Gender pay gap là gì?
A) ✅ Khoảng cách lương giữa nam và nữ  
B) Tổng lương công ty 
C) Lương trung bình 
D) Số lượng nhân viên

Giải thích: Gender pay gap đo sự chênh lệch lương trung bình giữa hai giới.

### Quiz 3: Query nào tính lương trung bình?
A) SELECT MAX(salary) FROM salaries 
B) ✅ SELECT AVG(salary) FROM salaries  
C) SELECT SUM(salary) FROM salaries 
D) SELECT COUNT(salary) FROM salaries

Giải thích: `AVG()` tính giá trị trung bình, phù hợp để tính lương trung bình.

### Quiz 4: Để nhóm dữ liệu theo phòng ban, dùng?
A) ORDER BY
B) WHERE
C) ✅ GROUP BY  
D) JOIN

Giải thích: `GROUP BY` dùng để nhóm dữ liệu theo cột như dept_name.

### Quiz 5: ROUND(1234.567, 1) trả về?
A) ✅ 1234.6  
B) 1235  
C) 1234.57  
D) 1234

Giải thích: `ROUND(1234.567,1)` làm tròn đến 1 chữ số thập phân -> 1234.6.

---


## 🎓 Kết luận Buổi 5

✅ Bạn đã hoàn thành:
- Dùng SELECT + JOIN + aggregate để phân tích lương
- Sử dụng ROUND/NUMERIC làm tròn trong báo cáo lương
- Phân tích phân bố lương theo phòng ban và cấp bậc
- Tính Gender Pay Gap và correlation salary vs experience
- Window function + CTEs: LAG để tính lịch sử tăng/giảm lương

📅 Buổi 6: Engagement & Advanced Analysis
- Phân tích Engagement và Wellbeing score
- Sử dụng window function (RANK, SUM OVER) và CTE nâng cao
- Xây bảng score, tính KPIs theo phòng ban, xác định nhóm nguy cơ

---

💡 Ghi chú rất quan trọng:
- Luôn kiểm tra `effective_date` khi dùng bảng salaries lịch sử.
- Dữ liệu phải có nhiều thời điểm (time-series) mới phân tích tăng/giảm chính xác.
- CTE + window function giúp query dễ đọc và mở rộng.

🚀 Tiếp theo: Mở file `Buoi6_ENGAGEMENT_ADVANCED_ANALYSIS.ipynb` và bắt đầu chinh phục nội dung mới nhé !